# ID2222 Homework 2
## A-Priori 频繁项集和关联规则

这个 Notebook 用来解释、测试和运行 `apriori.py`。

运行目标有两个：

1. 找到支持度不低于 `MIN_SUPPORT` 的频繁项集。
2. 如果设置了 `MIN_CONFIDENCE`，再生成关联规则。

这里尽量把每个步骤写成可以单独阅读的形式，方便在 TA 展示时解释程序怎样工作。

## 1. 题目中的两个指标

一条交易可以看成一个 basket，其中包含多个项目。

**Support count** 是包含某个 itemset 的交易数量。例如，`{A, B}` 出现在 50 条交易中，那么它的支持度就是 50。

**Confidence** 用于关联规则 `X -> Y`。它等于包含 `X union Y` 的交易数量除以包含 `X` 的交易数量。

本项目使用支持度数量作为 `min_support`，使用 0 到 1 之间的小数作为 `min_confidence`。

In [1]:
# 只使用 Python 标准库和项目自己的 apriori.py。
# Notebook 通常会从 TW2 启动，但下面也会尝试两个常见的上级目录，
# 这样从 Coding 或工作区目录启动 Jupyter 时也可以找到项目。
from pathlib import Path
import sys

candidate_project_dirs = [
    Path.cwd(),
    Path.cwd() / "TW2",
    Path.cwd().parent / "TW2",
]
PROJECT_DIR = next(
    (directory for directory in candidate_project_dirs if (directory / "apriori.py").exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "找不到 TW2 项目目录，请在 TW2、Coding 或其上级目录中启动 Notebook"
    )

from apriori import (
    format_itemset,
    generate_association_rules,
    read_transactions,
    run_apriori,
    write_results,
)

sys.path.insert(0, str(PROJECT_DIR))

print(f"当前工作目录：{Path.cwd()}")
print(f"项目目录：{PROJECT_DIR}")


当前工作目录：C:\Users\goahe\Desktop\ID2222\Coding\TW2
项目目录：C:\Users\goahe\Desktop\ID2222\Coding\TW2


## 2. 先用一个很小的数据集验证算法

下面的四条交易可以人工检查：

- A 出现 3 次
- B 出现 3 次
- C 出现 3 次
- AB、AC、BC 各出现 2 次
- ABC 只出现 1 次，所以在最小支持度为 2 时应被删除

In [2]:
# 这个小数据集不依赖外部文件，方便随时重复运行。
small_transactions = [
    frozenset({"A", "B", "C"}),
    frozenset({"A", "B"}),
    frozenset({"A", "C"}),
    frozenset({"B", "C"}),
]

small_result = run_apriori(small_transactions, min_support=2)
small_supports = small_result["support_counts"]

# 这些断言把人工计算的结果变成了自动检查。
assert small_supports[frozenset({"A"})] == 3
assert small_supports[frozenset({"A", "B"})] == 2
assert 3 not in small_result["frequent_itemsets"]

small_rules = generate_association_rules(
    small_result["frequent_itemsets"],
    small_supports,
    min_support=2,
    min_confidence=0.6,
)
assert small_rules, "小数据集应该至少产生一条规则"
print("小数据集测试通过")

小数据集测试通过


## 3. 选择完整数据集和实验参数

作业页面没有指定固定的 `s` 和 `c`。下面的值是一个可复现的示例：

- `MIN_SUPPORT = 1000` 表示至少出现在 1000 条交易中
- `MIN_CONFIDENCE = 0.6` 表示置信度至少为 60%

如果课程公告或 TA 给出了指定参数，只需要修改这两个变量。

In [3]:
# 数据集位于 TW2 的上一级目录 Coding。
DATASET_PATH = PROJECT_DIR.parent / "T10I4D100K.dat"

# 这两个参数是示例值，不是页面中指定的固定值。
MIN_SUPPORT = 1000
MIN_CONFIDENCE = 0.6

assert DATASET_PATH.exists(), f"找不到数据集：{DATASET_PATH}"
print(f"数据集路径：{DATASET_PATH}")

数据集路径：C:\Users\goahe\Desktop\ID2222\Coding\T10I4D100K.dat


## 4. 读取完整数据集

文件每行是一条交易，项目之间使用空格分隔。
`read_transactions` 会把每行转换为集合，因此一条交易中的重复项目不会重复计数。

In [4]:
transactions = read_transactions(DATASET_PATH)
print(f"交易数量：{len(transactions)}")
print(f"第一条交易：{format_itemset(transactions[0])}")
print(f"第一条交易的项目数：{len(transactions[0])}")

交易数量：100000
第一条交易：{164, 240, 25, 274, 328, 368, 448, 52, 538, 561, 630, 687, 730, 775, 825, 834}
第一条交易的项目数：16


## 5. 运行 A-Priori

算法从频繁 1-项集开始，逐层生成候选项集。每一层会进行两步：

1. 使用上一层频繁项集生成候选。
2. 扫描交易并统计候选的支持度。

输出中的 `level_timings` 可以帮助我们理解每一层花费了多少时间。

In [5]:
result = run_apriori(transactions, min_support=MIN_SUPPORT)
frequent_by_size = result["frequent_itemsets"]

print(f"总运行时间：{result['total_seconds']:.3f} 秒")
print(f"频繁项集总数：{sum(len(level) for level in frequent_by_size.values())}")

print("\n每一层的统计：")
for timing in result["level_timings"]:
    print(
        f"大小 {timing['itemset_size']} | "
        f"候选 {timing['candidate_count']} | "
        f"频繁 {timing['frequent_count']} | "
        f"耗时 {timing['seconds']:.3f} 秒"
    )

总运行时间：5.802 秒
频繁项集总数：385

每一层的统计：
大小 1 | 候选 870 | 频繁 375 | 耗时 0.274 秒
大小 2 | 候选 70125 | 频繁 9 | 耗时 2.560 秒
大小 3 | 候选 1 | 频繁 1 | 耗时 2.968 秒
大小 4 | 候选 0 | 频繁 0 | 耗时 0.000 秒


## 6. 查看部分频繁项集

为了避免 Notebook 输出过长，这里只展示每一层前几个结果。
CSV 文件会保存全部结果。

In [6]:
for itemset_size in sorted(frequent_by_size):
    print(f"\n{itemset_size}-项集")
    for itemset, support in list(frequent_by_size[itemset_size].items())[:10]:
        print(f"{format_itemset(itemset)} -> support {support}")


1-项集
{1} -> support 1535
{10} -> support 1351
{100} -> support 1749
{104} -> support 1158
{105} -> support 1100
{110} -> support 1801
{111} -> support 1171
{112} -> support 2680
{115} -> support 1775
{116} -> support 2193

2-项集
{217, 346} -> support 1336
{227, 390} -> support 1049
{368, 682} -> support 1193
{368, 829} -> support 1194
{39, 704} -> support 1107
{39, 825} -> support 1187
{390, 722} -> support 1042
{704, 825} -> support 1102
{789, 829} -> support 1194

3-项集
{39, 704, 825} -> support 1035


## 7. 生成可选的关联规则

只有大小至少为 2 的频繁项集才可能产生规则。
对于每个频繁项集，程序会枚举非空真子集作为前件，并计算置信度。

In [7]:
rules = generate_association_rules(
    result["frequent_itemsets"],
    result["support_counts"],
    min_support=MIN_SUPPORT,
    min_confidence=MIN_CONFIDENCE,
)

print(f"满足条件的关联规则数量：{len(rules)}")
for rule in rules[:20]:
    print(
        f"{format_itemset(rule['antecedent'])} -> "
        f"{format_itemset(rule['consequent'])} | "
        f"support {rule['support']} | "
        f"confidence {rule['confidence']:.3f}"
    )

满足条件的关联规则数量：5
{704, 825} -> {39} | support 1035 | confidence 0.939
{39, 704} -> {825} | support 1035 | confidence 0.935
{39, 825} -> {704} | support 1035 | confidence 0.872
{704} -> {39} | support 1107 | confidence 0.617
{704} -> {825} | support 1102 | confidence 0.614


## 8. 保存实验结果

保存结果后，报告可以直接引用 CSV 中的数据，`run_summary.json` 可以提供运行时间和数量统计。
数据集文件本身不复制到 `results`，也不应该放入最终提交的 zip。

In [8]:
OUTPUT_DIR = PROJECT_DIR / "results_notebook"
write_results(OUTPUT_DIR, result, rules)
print(f"结果已保存到：{OUTPUT_DIR.resolve()}")

结果已保存到：C:\Users\goahe\Desktop\ID2222\Coding\TW2\results_notebook


## 9. 报告和展示时需要说明的内容

- A-Priori 的候选生成和子集剪枝过程
- 支持度的定义和本次实验使用的阈值
- 数据集来源和获取方式
- 频繁项集数量和示例
- 每一层和总运行时间
- 关联规则的支持度和置信度，如果完成可选任务
- 如何运行 `apriori.py`
- AI Disclosure statement

这份 Notebook 可以作为报告的实验部分，也可以导出为 PDF。提交时仍需按照 Canvas 要求一并放入源代码、运行脚本和报告。

## 10. AI Disclosure statement

本项目使用生成式 AI 协助理解作业要求、辅助梳理算法结构、整理文档和补充注释。最终代码由学生在本地数据集上运行、测试和检查，实验结果来自本地实际运行。数据集选择、参数选择、结果解释和最终提交内容由学生负责。